Clonar o repositorio

In [1]:
!git clone https://github.com/carbon-footprint-analysis/carbon-footprint-analysis.git

fatal: destination path 'carbon-footprint-analysis' already exists and is not an empty directory.


Instalar a versão especifica para o projeto (VsCode x Colab)

In [2]:
pip install scikit-learn==1.8.0

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


Importar as bibliotecas

In [3]:
import joblib
import urllib.request
import pandas as pd

Importando o arquivo joblib

In [4]:
import os

# Ver onde você está agora
print("Diretório atual:", os.getcwd())

# Procurar o arquivo .joblib em todo o projeto
for root, dirs, files in os.walk('.'):
    for file in files:
        if file.endswith('.joblib'):
            print("Encontrado:", os.path.join(root, file))

Diretório atual: c:\Repositorio\carbon-footprint-analysis\notebooks


In [5]:
import os

# Detecta automaticamente o caminho correto do modelo
# Funciona tanto localmente (VS Code / Jupyter) quanto no Google Colab

def find_model_path():
    candidates = [
        # Caminho relativo (execucao local a partir de notebooks/)
        os.path.join('..', 'models', 'best_carbon_footprint_model.joblib'),
        # Caminho relativo (execucao a partir da raiz do projeto)
        os.path.join('models', 'best_carbon_footprint_model.joblib'),
        # Caminho absoluto Google Colab
        '/content/carbon-footprint-analysis/models/best_carbon_footprint_model.joblib',
        # Caminho absoluto baseado no diretorio atual
        os.path.join(os.getcwd(), '..', 'models', 'best_carbon_footprint_model.joblib'),
    ]
    for path in candidates:
        if os.path.exists(path):
            return os.path.abspath(path)
    return None

model_path = find_model_path()

if model_path:
    model = joblib.load(model_path)
    print(f'Modelo carregado com sucesso!')
    print(f'Caminho: {model_path}')
else:
    print('ERRO: Modelo nao encontrado.')
    print('Certifique-se de que o notebook 04 foi executado e o arquivo')
    print('best_carbon_footprint_model.joblib esta na pasta models/')
    print()
    print('Diretorio atual:', os.getcwd())
    print('Conteudo do diretorio pai:')
    parent = os.path.join(os.getcwd(), '..')
    for item in os.listdir(parent):
        print(' ', item)


Modelo carregado com sucesso!
Caminho: c:\Repositorio\carbon-footprint-analysis\models\best_carbon_footprint_model.joblib


Analisando o modelo para sanity

In [6]:
print(model)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['consumo_kwh', 'mes']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['estado', 'setor',
                                                   'fonte_energia',
                                                   'season'])])),
                ('regressor',
                 RandomForestRegressor(max_depth=10, n_jobs=-1,
                                       random_state=42))])


In [7]:
encoder = model.named_steps['preprocessor'].named_transformers_['cat']

encoder.categories_

[array(['AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MG',
        'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RR',
        'RS', 'SC', 'SE', 'SP', 'TO'], dtype=object),
 array(['comercial', 'industrial', 'outros', 'residencial', 'rural'],
       dtype=object),
 array(['eólica', 'hidrelétrica', 'nuclear', 'solar', 'térmica'],
       dtype=object),
 array(['Inverno', 'Outono', 'Primavera', 'Verao'], dtype=object)]

Testando o modelo

In [8]:
df = pd.DataFrame({
    "consumo_kwh": [1200],
    "mes": [6],
    "estado": ["SP"],
    "setor": ["industrial"],
    "fonte_energia": ["hydro"],
    "season": ["Inverno"]
})

model.predict(df)

array([13.4938226])

Fazendo o Wrapper


Utilizando o modelo para fazer o comparativo entre todas as fontes de energia eletrica

In [9]:
def predict_all_sources(model, energy_kwh, month, state, usage_type, season):
    sources = ['hidrelétrica', 'nuclear', 'solar', 'térmica', 'eólica']

    results = {}

    for source in sources:

        df = pd.DataFrame({
            "consumo_kwh": [energy_kwh],
            "mes": [month],
            "estado": [state],
            "setor": [usage_type],
            "fonte_energia": [source],
            "season": [season]
        })

        co2 = model.predict(df)[0]

        results[source] = round(float(co2), 2)

    results_sorted = dict(sorted(results.items(), key=lambda x: x[1]))

    return results_sorted

In [10]:
predict_all_sources(
    model,
    energy_kwh=1200,
    month=6,
    state="SP",
    usage_type="industrial",
    season="Inverno"
)

{'hidrelétrica': 4.84,
 'nuclear': 13.49,
 'eólica': 13.49,
 'solar': 54.23,
 'térmica': 969.09}

Criando comparativo para combustiveis liquidos foi utilizado a media de motores motores modernos. E os valores foram retirados do `PCC Guidelines for National Greenhouse Gas Inventories`, `IEA – International Energy Agency` e `US EPA emission factors`

In [11]:
def liquid_fuel_emissions(energy_kwh):

    fuels = {
        "ethanol": {
            "efficiency": 0.27,
            "emission_factor": 0.20
        },
        "gasoline": {
            "efficiency": 0.30,
            "emission_factor": 0.64
        },
        "diesel": {
            "efficiency": 0.38,
            "emission_factor": 0.73
        }
    }

    results = {}

    for fuel, data in fuels.items():

        efficiency = data["efficiency"]
        factor = data["emission_factor"]

        energy_input = energy_kwh / efficiency
        co2 = energy_input * factor

        results[fuel] = round(co2, 2)

    results_sorted = dict(sorted(results.items(), key=lambda x: x[1]))

    return results_sorted

In [12]:
liquid_fuel_emissions(1200)

{'ethanol': 888.89, 'diesel': 2305.26, 'gasoline': 2560.0}

Unificando fontes geradoras e ordenando do menos poluente para o mais poluente adicionado a % de emissão de Co2 que foi aumentado ou diminuido se comparado com a fonte geradora base do brasil `Hydro`

In [13]:
def compare_energy_sources(model, energy_kwh, month, state, usage_type, season):

    electricity = predict_all_sources(
        model,
        energy_kwh,
        month,
        state,
        usage_type,
        season
    )

    fuels = liquid_fuel_emissions(energy_kwh)

    combined = {**electricity, **fuels}

    ranking = dict(sorted(combined.items(), key=lambda x: x[1]))

    base = ranking["hidrelétrica"]

    result = {}

    for source, value in ranking.items():

        percent = ((value - base) / base) * 100

        result[source] = {
            "co2": round(value, 2),
            "vs_hydro_%": round(percent, 2)
        }

    return result

In [14]:
compare_energy_sources(
    model,
    energy_kwh=1200,
    month=6,
    state="SP",
    usage_type="industrial",
    season="Inverno"
)

{'hidrelétrica': {'co2': 4.84, 'vs_hydro_%': 0.0},
 'nuclear': {'co2': 13.49, 'vs_hydro_%': 178.72},
 'eólica': {'co2': 13.49, 'vs_hydro_%': 178.72},
 'solar': {'co2': 54.23, 'vs_hydro_%': 1020.45},
 'ethanol': {'co2': 888.89, 'vs_hydro_%': 18265.5},
 'térmica': {'co2': 969.09, 'vs_hydro_%': 19922.52},
 'diesel': {'co2': 2305.26, 'vs_hydro_%': 47529.34},
 'gasoline': {'co2': 2560.0, 'vs_hydro_%': 52792.56}}

CheckPonit Wrapper

In [15]:
!pip install scikit-learn==1.8.0

!git clone https://github.com/carbon-footprint-analysis/carbon-footprint-analysis.git

import sys
import os

# Funciona tanto no Colab quanto localmente
colab_path = "/content/carbon-footprint-analysis"
local_path = os.path.join(os.path.dirname(os.getcwd()), '')

if os.path.exists(colab_path):
    sys.path.append(colab_path)
else:
    sys.path.append(local_path)

Defaulting to user installation because normal site-packages is not writeable


fatal: destination path 'carbon-footprint-analysis' already exists and is not an empty directory.


Criando .py do wrapper

Versão Final wrapper

importando o wrapper diretamente do .py para possivel implementação no FastAPI

In [16]:
!pip install scikit-learn==1.8.0

!git clone https://github.com/carbon-footprint-analysis/carbon-footprint-analysis.git

import sys
import os

# Funciona tanto no Colab quanto localmente
colab_path = "/content/carbon-footprint-analysis"
local_path = os.path.join(os.path.dirname(os.getcwd()), '')

if os.path.exists(colab_path):
    sys.path.append(colab_path)
else:
    sys.path.append(local_path)

Defaulting to user installation because normal site-packages is not writeable


fatal: destination path 'carbon-footprint-analysis' already exists and is not an empty directory.


In [17]:
wrapper_code = '''import joblib
import pandas as pd
import os

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
model = joblib.load(os.path.join(BASE_DIR, "models", "best_carbon_footprint_model.joblib"))


def predict_all_sources(model, energy_kwh, month, state, usage_type, season):
    sources = ['hidrelétrica', 'nuclear', 'solar', 'térmica', 'eólica']
    results = {}

    for source in sources:
        df = pd.DataFrame({
            "consumo_kwh": [energy_kwh],
            "mes": [month],
            "estado": [state],
            "setor": [usage_type],
            "fonte_energia": [source],
            "season": [season]
        })
        co2 = model.predict(df)[0]
        results[source] = round(float(co2), 2)

    return dict(sorted(results.items(), key=lambda x: x[1]))


def liquid_fuel_emissions(energy_kwh):
    fuels = {
        "ethanol":  {"efficiency": 0.27, "emission_factor": 0.20},
        "gasoline": {"efficiency": 0.30, "emission_factor": 0.64},
        "diesel":   {"efficiency": 0.38, "emission_factor": 0.73}
    }
    results = {}
    for fuel, data in fuels.items():
        energy_input = energy_kwh / data["efficiency"]
        co2 = energy_input * data["emission_factor"]
        results[fuel] = round(co2, 2)

    return dict(sorted(results.items(), key=lambda x: x[1]))


def compare_energy_sources(energy_kwh, month, state, usage_type, season):
    electricity = predict_all_sources(model, energy_kwh, month, state, usage_type, season)
    fuels = liquid_fuel_emissions(energy_kwh)
    combined = {**electricity, **fuels}
    ranking = dict(sorted(combined.items(), key=lambda x: x[1]))
    base = ranking["hidrelétrica"]
    result = {}
    for source, value in ranking.items():
        percent = ((value - base) / base) * 100
        result[source] = {"co2": round(value, 2), "vs_hydro_%": round(percent, 2)}
    return result
'''

with open("wrapper.py", "w", encoding="utf-8") as f:
    f.write(wrapper_code)

print("wrapper.py salvo!")

wrapper.py salvo!


In [18]:
# Importar o wrapper (funciona local e no Colab)
import sys
import os

# Adiciona a raiz do projeto ao path para importar wrapper.py
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from wrapper import compare_energy_sources
print('wrapper.py importado com sucesso!')


wrapper.py importado com sucesso!


Diversos testes para verificar confiabilidade do wrapper

In [19]:
print(compare_energy_sources(1200, 6,  "SP", "industrial",  "Inverno"))
print(compare_energy_sources(800,  3,  "RJ", "comercial",   "Outono"))     # commercial → comercial
print(compare_energy_sources(1500, 12, "MG", "residencial", "Verao"))      # Verão → Verao
print(compare_energy_sources(600,  9,  "RS", "industrial",  "Primavera"))
print(compare_energy_sources(2000, 1,  "BA", "comercial",   "Verao"))      # commercial → comercial, Verão → Verao
print(compare_energy_sources(400,  7,  "SC", "residencial", "Inverno"))
print(compare_energy_sources(1000, 5,  "GO", "rural",       "Outono"))
print(compare_energy_sources(1800, 10, "PA", "industrial",  "Primavera"))
print(compare_energy_sources(500,  4,  "CE", "comercial",   "Outono"))     # commercial → comercial
print(compare_energy_sources(2200, 8,  "PR", "industrial",  "Inverno"))

{'Hidrelétrica': {'co2': 13.49, 'vs_hydro_pct': 0.0}, 'Nuclear': {'co2': 13.49, 'vs_hydro_pct': 0.0}, 'Solar': {'co2': 13.49, 'vs_hydro_pct': 0.0}, 'Térmica': {'co2': 13.49, 'vs_hydro_pct': 0.0}, 'Eólica': {'co2': 13.49, 'vs_hydro_pct': 0.0}, 'Etanol': {'co2': 888.89, 'vs_hydro_pct': 6489.25}, 'Diesel': {'co2': 2305.26, 'vs_hydro_pct': 16988.66}, 'Gasolina': {'co2': 2560.0, 'vs_hydro_pct': 18877.02}}
{'Hidrelétrica': {'co2': 8.73, 'vs_hydro_pct': 0.0}, 'Nuclear': {'co2': 8.73, 'vs_hydro_pct': 0.0}, 'Solar': {'co2': 8.73, 'vs_hydro_pct': 0.0}, 'Térmica': {'co2': 8.73, 'vs_hydro_pct': 0.0}, 'Eólica': {'co2': 8.73, 'vs_hydro_pct': 0.0}, 'Etanol': {'co2': 592.59, 'vs_hydro_pct': 6687.97}, 'Diesel': {'co2': 1536.84, 'vs_hydro_pct': 17504.12}, 'Gasolina': {'co2': 1706.67, 'vs_hydro_pct': 19449.48}}
{'Hidrelétrica': {'co2': 16.46, 'vs_hydro_pct': 0.0}, 'Nuclear': {'co2': 16.46, 'vs_hydro_pct': 0.0}, 'Solar': {'co2': 16.46, 'vs_hydro_pct': 0.0}, 'Térmica': {'co2': 16.46, 'vs_hydro_pct': 0.0}, 

In [20]:
# Ver a importância de cada feature no modelo
import pandas as pd
import matplotlib.pyplot as plt

feature_names = (
    list(model.named_steps['preprocessor']
         .named_transformers_['num']
         .get_feature_names_out(['consumo_kwh', 'mes']))
    +
    list(model.named_steps['preprocessor']
         .named_transformers_['cat']
         .get_feature_names_out(['estado', 'setor', 'fonte_energia', 'season']))
)

importances = model.named_steps['regressor'].feature_importances_

fi = pd.DataFrame({'feature': feature_names, 'importance': importances})
fi = fi.sort_values('importance', ascending=False)

print(fi.to_string(index=False))

                   feature   importance
               consumo_kwh 7.565703e-01
     fonte_energia_térmica 2.417373e-01
       fonte_energia_solar 6.722201e-04
                       mes 2.920192e-04
                 estado_PA 6.235989e-05
                 estado_SP 5.708683e-05
             season_Outono 4.954133e-05
                 estado_SC 4.520450e-05
                 estado_RJ 4.332509e-05
                 estado_BA 3.955932e-05
          season_Primavera 3.890256e-05
                 estado_MG 3.778552e-05
fonte_energia_hidrelétrica 3.709893e-05
            season_Inverno 3.649570e-05
              season_Verao 3.516209e-05
                 estado_PR 3.423427e-05
                 estado_RS 3.205772e-05
                 estado_AM 2.547847e-05
                 estado_GO 2.232429e-05
                 estado_PE 1.845503e-05
                 estado_MS 1.689143e-05
                 estado_MA 1.642095e-05
                 estado_PB 1.274538e-05
                 estado_CE 8.928990e-06


In [21]:
import os

for root, dirs, files in os.walk("/content/carbon-footprint-analysis"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

In [22]:
import os
import pandas as pd

# Pega o caminho de onde o notebook está e sobe um nível
BASE_DIR = os.path.dirname(os.getcwd()) 
DATA_PATH = os.path.join(BASE_DIR, "data", "processed", "synthetic_energy_emissions_dataset.csv")

df = pd.read_csv(DATA_PATH)

In [23]:
# Ver o CO₂ médio normalizado por consumo
df['co2_por_kwh'] = df['emissao_co2'] / df['consumo_kwh']
print(df.groupby('fonte_energia')['co2_por_kwh'].mean().sort_values())

fonte_energia
hidrelétrica    0.003997
eólica          0.011004
nuclear         0.011994
solar           0.044951
térmica         0.819658
Name: co2_por_kwh, dtype: float64


In [24]:
import os
import pandas as pd

# Verifica se a pasta do Colab existe, se não, usa o caminho relativo local
if os.path.exists('/content/carbon-footprint-analysis'):
    path = '/content/carbon-footprint-analysis/data/processed/v2_energy_source_emission_factors.csv'
else:
    path = '../data/processed/v2_energy_source_emission_factors.csv'

emission_df = pd.read_csv(path)